In [ ]:
# Run once in Colab
!pip install -q wandb pandas

In [ ]:
import wandb
wandb.login()

In [ ]:
import re
import pandas as pd

ENTITY  = "arc_agi"
PROJECT = "slm-instability"

api  = wandb.Api()
runs = api.runs(f"{ENTITY}/{PROJECT}")

rows = []
for run in runs:
    name = run.name or ""
    if not name.startswith("lr-sweep-"):
        continue

    # parse LR and model size from run name: lr-sweep-{model}-lr{lr}
    # e.g. lr-sweep-42M-lr3e-2  ->  model=42M, lr=3e-2
    m = re.search(r"lr-sweep-(.+)-lr(.+)$", name)
    if not m:
        continue
    model_name = m.group(1)   # e.g. 2p4M, 9p4M, 19M, 42M
    lr         = float(m.group(2))

    # best val loss: minimum across the run history
    # (W&B summary stores the *last* value, not the best)
    hist = run.history(keys=["eval/val_loss"], pandas=True)
    best_val_loss = hist["eval/val_loss"].min() if "eval/val_loss" in hist.columns else None

    rows.append({
        "run_name":      name,
        "model":         model_name,
        "lr":            lr,
        "state":         run.state,
        "best_val_loss": best_val_loss,
        "max_steps":     run.config.get("trainer", {}).get("max_steps"),
    })

df = pd.DataFrame(rows).sort_values(["model", "lr"]).reset_index(drop=True)
print(df)

In [ ]:
# if the eval metric key is different, check what's available:
# run = api.runs(f"{ENTITY}/{PROJECT}")[0]
# print(list(run.history().columns))